In [1]:
from transformers import pipeline

pipe = pipeline("text-generation", model="google/gemma-3-270m-it", device='mps')
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

Device set to use mps


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': 'I am Gemma, an open-weights AI assistant. I am a large language model created by the Gemma team.\n'}]}]

In [2]:
from inference_tool import GemmaModel, Tokenizer, generate, format_messages

model = GemmaModel("google/gemma-3-270m-it")
tokenizer = Tokenizer("google/gemma-3-270m-it")

prompt = format_messages([{"role": "user", "content": "Who are you?"}])

tokens = tokenizer.encode(prompt)
output = generate(model, tokenizer, tokens, max_new_tokens=30, temperature=0.0)
print(tokenizer.decode(output))

user
Who are you?
model
I am Gemma, an open-weights AI assistant. I am a large language model created by the Gemma team at Google DeepMind.



In [3]:
# ============================================================================
# STEP-BY-STEP: How text generation works in a Gemma model
# ============================================================================

import numpy as np
from inference_tool import GemmaModel, Tokenizer, format_messages

# --- Step 0: Load model and tokenizer ---
# The model loads pre-trained weights from Hugging Face:
#   - Token embeddings: maps token IDs to dense vectors (vocab_size x hidden_size)
#   - Transformer layers: attention + MLP blocks that process sequences
#   - LM head: projects hidden states back to vocabulary logits
model = GemmaModel("google/gemma-3-270m-it")
tokenizer = Tokenizer("google/gemma-3-270m-it")

In [4]:
# --- Step 1: Format and tokenize the prompt ---
# Chat models require special formatting with role markers and turn tokens
prompt = format_messages([{"role": "user", "content": "Who are you?"}])
print("Formatted prompt:")
print(repr(prompt))

# Tokenize: convert text to token IDs
prompt_tokens = tokenizer.encode(prompt)
print(f"\nPrompt tokens: {prompt_tokens}")
print(f"Token count: {len(prompt_tokens)}")

Formatted prompt:
'<start_of_turn>user\nWho are you?<end_of_turn>\n<start_of_turn>model\n'

Prompt tokens: [105, 2364, 107, 15938, 659, 611, 236881, 106, 107, 105, 4368, 107]
Token count: 12


In [5]:
# --- Step 2: Add BOS token if needed ---
# Gemma models require a Beginning-of-Sequence token at the start
bos_token_id = tokenizer.bos_token_id
if prompt_tokens[0] != bos_token_id:
    prompt_tokens = [bos_token_id] + list(prompt_tokens)
    print(f"\nAdded BOS token: {prompt_tokens}")

# Initialize output sequence with prompt tokens
output_tokens = list(prompt_tokens)


Added BOS token: [2, 105, 2364, 107, 15938, 659, 611, 236881, 106, 107, 105, 4368, 107]


In [6]:
# ============================================================================
# PREFILL PHASE: Process the entire prompt in one forward pass
# ============================================================================

# --- Step 3: Convert tokens to input tensor ---
# Shape: (batch_size=1, seq_len=N)
token_ids = np.array([prompt_tokens], dtype=np.int32)
print(f"\nInput shape: {token_ids.shape}")


Input shape: (1, 13)


In [7]:
# --- Step 4: Forward pass through the model ---
# The model processes all tokens at once.
#
# Inside model.forward():
#   a) Embedding lookup: token IDs → dense vectors (scaled by √hidden_size)
#   b) Transformer layers (×18 for Gemma-3-270M):
#      - input_layernorm (RMSNorm) → Self-Attention (with RoPE + Q/K norm) 
#        → post_attention_layernorm (RMSNorm) → residual add
#      - pre_feedforward_layernorm (RMSNorm) → MLP (GEGLU with GELU activation) 
#        → post_feedforward_layernorm (RMSNorm) → residual add
#   c) Final RMSNorm
#   d) LM head projection: hidden states → vocabulary logits
#
# Returns:
#   - logits: probability scores for each token in vocabulary (1, seq_len, vocab_size)
#   - kv_cache: stored key/value tensors for efficient decoding (one per layer)
print("\n📍 PREFILL: Running forward pass on entire prompt...")
logits, kv_cache = model.forward(token_ids, position_offset=0, kv_cache=None)
print(f"Logits shape: {logits.shape}")  # (1, seq_len, vocab_size)
print(f"KV cache: {len(kv_cache)} layers")


📍 PREFILL: Running forward pass on entire prompt...
Logits shape: (1, 13, 262144)
KV cache: 18 layers


In [8]:
# --- Step 5: Extract logits for the last position ---
# We only care about the last token's predictions (what comes next?)
next_token_logits = logits[0, -1, :].copy()
print(f"\nLast position logits shape: {next_token_logits.shape}")  # (vocab_size,)

# Show top 5 candidate tokens
def show_top_tokens(logits, tokenizer, k=5):
    top_indices = np.argsort(logits)[-k:][::-1]
    print("Top predicted tokens:")
    for idx in top_indices:
        token_id = int(idx)  # Convert numpy int to Python int for tokenizer
        print(f"  {token_id}: '{tokenizer.decode([token_id])}' (logit: {logits[idx]:.2f})")

show_top_tokens(next_token_logits, tokenizer)


Last position logits shape: (262144,)
Top predicted tokens:
  236777: 'I' (logit: 23.42)
  106: '' (logit: 13.36)
  2205: 'As' (logit: 13.35)
  40281: 'मैं' (logit: 11.58)
  9259: 'Hello' (logit: 11.13)


In [9]:
# --- Step 6: Sample the next token (greedy with temperature=0) ---
# Temperature=0 means we just take argmax (most likely token)
def sample_greedy(logits):
    return int(np.argmax(logits))

first_new_token = sample_greedy(next_token_logits)
output_tokens.append(first_new_token)
print(f"\n🎯 Sampled token: {first_new_token} = '{tokenizer.decode([first_new_token])}'");


🎯 Sampled token: 236777 = 'I'


In [10]:
# ============================================================================
# DECODE PHASE: Generate tokens one at a time using KV cache
# ============================================================================

print("\n📍 DECODE: Generating tokens one by one...")

# Define stop tokens
stop_tokens = {tokenizer.eos_token_id, 106}  # 106 = <end_of_turn>
max_new_tokens = 30

for step in range(max_new_tokens - 1):
    # --- Step 7: Prepare input for next token ---
    # Only pass the LAST generated token (not the whole sequence!)
    # The KV cache stores previous tokens' key/value pairs
    current_token = output_tokens[-1]
    token_ids = np.array([[current_token]], dtype=np.int32)
    
    # Position offset tells the model where we are in the sequence
    # (used for RoPE positional encoding)
    position_offset = len(output_tokens) - 1
    
    # --- Step 8: Forward pass with KV cache ---
    # This is FAST because:
    #   - We only process 1 token instead of the whole sequence
    #   - Attention uses cached keys/values from previous positions
    #   - Only need to compute attention for new token against all previous
    logits, kv_cache = model.forward(
        token_ids, 
        position_offset=position_offset, 
        kv_cache=kv_cache
    )
    
    # --- Step 9: Sample next token ---
    next_token_logits = logits[0, -1, :].copy()
    next_token = sample_greedy(next_token_logits)
    output_tokens.append(next_token)
    
    # Show progress
    decoded_so_far = tokenizer.decode([next_token])
    print(f"  Step {step+1}: token={next_token:>5}, text='{decoded_so_far}'")
    
    # --- Step 10: Check for stop condition ---
    if next_token in stop_tokens:
        print(f"\n🛑 Stop token reached!")
        break


📍 DECODE: Generating tokens one by one...
  Step 1: token= 1006, text=' am'
  Step 2: token=147224, text=' Gemma'
  Step 3: token=236764, text=','
  Step 4: token=  614, text=' an'
  Step 5: token= 1932, text=' open'
  Step 6: token=236772, text='-'
  Step 7: token=38357, text='weights'
  Step 8: token=12498, text=' AI'
  Step 9: token=16326, text=' assistant'
  Step 10: token=236761, text='.'
  Step 11: token=  564, text=' I'
  Step 12: token= 1006, text=' am'
  Step 13: token=  496, text=' a'
  Step 14: token= 2455, text=' large'
  Step 15: token= 5192, text=' language'
  Step 16: token= 2028, text=' model'
  Step 17: token= 4464, text=' created'
  Step 18: token=  684, text=' by'
  Step 19: token=  506, text=' the'
  Step 20: token=147224, text=' Gemma'
  Step 21: token= 2434, text=' team'
  Step 22: token=  657, text=' at'
  Step 23: token= 6475, text=' Google'
  Step 24: token=22267, text=' Deep'
  Step 25: token=65153, text='Mind'
  Step 26: token=236761, text='.'
  Step 27: tok

In [11]:
# ============================================================================
# FINAL OUTPUT
# ============================================================================
print("\n" + "="*60)
print("FINAL GENERATED TEXT:")
print("="*60)
print(tokenizer.decode(output_tokens))


FINAL GENERATED TEXT:
user
Who are you?
model
I am Gemma, an open-weights AI assistant. I am a large language model created by the Gemma team at Google DeepMind.

